1. Multi-table join: track name + album title + artist name (joining Track → Album → Artist)
2. Subquery: customers who spent more than the average total invoice amount
3. Window function: rank tracks within each genre by unit price (RANK() or ROW_NUMBER())

In [2]:
import os
os.chdir("c:\\Users\\Tarek\\OneDrive\\Documents\\data-analytics-course-2026")

In [3]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('data/raw/Chinook_Sqlite.sqlite')

In [4]:
# Retrieve the tables from the database
query = """
SELECT name FROM sqlite_master WHERE type='table';
"""
pd.read_sql_query(query, conn)    # List all tables in the database

,name
0,Album
1,Artist
2,Customer
3,Employee
4,Genre
5,Invoice
6,InvoiceLine
7,MediaType
8,Playlist
9,PlaylistTrack


**1. Multi-table join Using Inner Join**

In [5]:
# Retrieve track name, album title, and artist name.
# Use LIMIT statement to include only the first 10 records.
# Use the inner join to join the 3 tables (Track, Album, and Artist)

query = """
SELECT t.Name as Track_Name, a.Title as Album_Title, ar.Name as Artist_Name
FROM Track t
INNER JOIN Album a ON t.AlbumId = a.AlbumId
INNER JOIN Artist ar ON a.ArtistId = ar.ArtistId
LIMIT 10;
"""
pd.read_sql_query(query, conn)

,Track_Name,Album_Title,Artist_Name
0,For Those About To Rock (We Salute You),For Those About To Rock We Salute You,AC/DC
1,Balls to the Wall,Balls to the Wall,Accept
2,Fast As a Shark,Restless and Wild,Accept
3,Restless and Wild,Restless and Wild,Accept
4,Princess of the Dawn,Restless and Wild,Accept
5,Put The Finger On You,For Those About To Rock We Salute You,AC/DC
6,Let's Get It Up,For Those About To Rock We Salute You,AC/DC
7,Inject The Venom,For Those About To Rock We Salute You,AC/DC
8,Snowballed,For Those About To Rock We Salute You,AC/DC
9,Evil Walks,For Those About To Rock We Salute You,AC/DC


**2. Subquery**

Find all customers who spent more than the average total invoice amount.

In [19]:
query = """
WITH customer_total_invoice AS (
        SELECT CustomerId, total_amount
        FROM (
            SELECT CustomerId, SUM(Total) AS total_amount 
            FROM Invoice
            GROUP BY CustomerId
            ) AS total_invoice
)

SELECT CustomerId, total_amount
FROM customer_total_invoice
WHERE total_amount > (SELECT AVG(total_amount)
                      FROM customer_total_invoice)
ORDER BY total_amount DESC;
"""

pd.read_sql_query(query, conn)

,CustomerId,total_amount
0,6,49.62
1,26,47.62
2,57,46.62
3,45,45.62
4,46,45.62
5,24,43.62
6,28,43.62
7,37,43.62
8,7,42.62
9,25,42.62


In [24]:
# 2nd way: Using nested subquery
query = """
SELECT i.CustomerId, 
       c.FirstName || ' ' || c.LastName AS Name,
       SUM(i.Total) AS Total_Spent
FROM Invoice i
JOIN Customer c ON i.CustomerId = c.CustomerId
GROUP BY i.CustomerId, c.FirstName, c.LastName
HAVING Total_Spent > (
                    SELECT AVG(Total_Spent) 
                    FROM 
                    (SELECT CustomerId, 
                            SUM(Total) AS Total_Spent 
                    FROM Invoice
                    GROUP BY CustomerId)
                    )
ORDER BY Total_Spent DESC;
"""
pd.read_sql_query(query, conn)

,CustomerId,Name,Total_Spent
0,6,Helena Holý,49.62
1,26,Richard Cunningham,47.62
2,57,Luis Rojas,46.62
3,45,Ladislav Kovács,45.62
4,46,Hugh O'Reilly,45.62
5,24,Frank Ralston,43.62
6,28,Julia Barnett,43.62
7,37,Fynn Zimmermann,43.62
8,7,Astrid Gruber,42.62
9,25,Victor Stevens,42.62


**3. Window Functions**

Rank tracks within each genre by unit price (RANK() or ROW_NUMBER())

In [25]:
query = """
SELECT * FROM Genre
LIMIT 10;
"""

pd.read_sql_query(query, conn)

,GenreId,Name
0,1,Rock
1,2,Jazz
2,3,Metal
3,4,Alternative & Punk
4,5,Rock And Roll
5,6,Blues
6,7,Latin
7,8,Reggae
8,9,Pop
9,10,Soundtrack


In [39]:
query = """
SELECT * FROM Track
LIMIT 20;
"""

pd.read_sql_query(query, conn)

,TrackId,Name,AlbumId,MediaTypeId,GenreId,Composer,Milliseconds,Bytes,UnitPrice
0,1,For Those About To Rock (We Salute You),1,1,1,"Angus Young, Malcolm Young, Brian Johnson",343719,11170334,0.99
1,2,Balls to the Wall,2,2,1,NaN,342562,5510424,0.99
2,3,Fast As a Shark,3,2,1,"F. Baltes, S. Kaufman, U. Dirkscneider & W. Ho...",230619,3990994,0.99
3,4,Restless and Wild,3,2,1,"F. Baltes, R.A. Smith-Diesel, S. Kaufman, U. D...",252051,4331779,0.99
4,5,Princess of the Dawn,3,2,1,Deaffy & R.A. Smith-Diesel,375418,6290521,0.99
5,6,Put The Finger On You,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",205662,6713451,0.99
6,7,Let's Get It Up,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",233926,7636561,0.99
7,8,Inject The Venom,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",210834,6852860,0.99
8,9,Snowballed,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",203102,6599424,0.99
9,10,Evil Walks,1,1,1,"Angus Young, Malcolm Young, Brian Johnson",263497,8611245,0.99


In [60]:
query = """
SELECT t.Name AS track_name,
       g.Name AS genre,
       UnitPrice,
       ROW_NUMBER() OVER(PARTITION BY g.Name ORDER BY UnitPrice) AS Row_N
FROM Genre g
JOIN Track t
ON g.GenreId = t.GenreId
LIMIT 50;
"""

pd.read_sql_query(query, conn)

,track_name,genre,UnitPrice,Row_N
0,War Pigs,Alternative,0.99,1
1,Say Hello 2 Heaven,Alternative,0.99,2
2,Reach Down,Alternative,0.99,3
3,Hunger Strike,Alternative,0.99,4
4,Pushin Forward Back,Alternative,0.99,5
5,Call Me a Dog,Alternative,0.99,6
6,Times of Trouble,Alternative,0.99,7
7,Wooden Jesus,Alternative,0.99,8
8,Your Savior,Alternative,0.99,9
9,Four Walled World,Alternative,0.99,10


In [57]:
# Use RANK()
query = """
SELECT t.Name AS track_name,
       g.Name AS genre,
       UnitPrice,
       RANK() OVER(PARTITION BY g.Name ORDER BY UnitPrice DESC) AS rank
FROM Genre g
JOIN Track t
ON g.GenreId = t.GenreId
ORDER BY rank DESC
;
"""
pd.read_sql_query(query, conn)

,track_name,genre,UnitPrice,rank
0,War Pigs,Alternative,0.99,1
1,Say Hello 2 Heaven,Alternative,0.99,1
2,Reach Down,Alternative,0.99,1
3,Hunger Strike,Alternative,0.99,1
4,Pushin Forward Back,Alternative,0.99,1
...,...,...,...,...
3498,No Clima,World,0.99,1
3499,A Moça e a Chuva,World,0.99,1
3500,Demorou!,World,0.99,1
3501,Din Din Wo (Little Child),World,0.99,1
